# Foundation Verification

Comprehensive verification of the SYK model simulator against known theoretical predictions.

## Contents
1. Level spacing ratio statistics (random matrix theory)
2. Thermal two-point function (conformal regime)
3. OTOC and Lyapunov exponent (chaos bound)
4. TFD entanglement entropy
5. Spectral form factor (dip-ramp-plateau)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import os

from src.syk import SYKHamiltonian
from src.doubled import DoubledSYK
from src.tfd import build_tfd, entanglement_entropy
from src.observables import level_spacing_ratio, spectral_form_factor, extract_lyapunov, thermal_two_point_vectorized
from src.disorder_average import save_results, load_results, check_cache
from src.utils import ensure_dir

ensure_dir('../results')
ensure_dir('../data')

plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'legend.fontsize': 10,
    'figure.figsize': (8, 6),
    'figure.dpi': 150,
})

print('Foundation verification notebook loaded.')

## 1. Level Spacing Ratio Statistics

The SYK model at q=4 is maximally chaotic and should exhibit random matrix theory (RMT) level statistics.
The level spacing ratio $\langle r \rangle$ should match the GOE/GUE prediction depending on the symmetry class.

- GOE: $\langle r \rangle \approx 0.5307$
- GUE: $\langle r \rangle \approx 0.6027$
- Poisson: $\langle r \rangle \approx 0.3863$

The symmetry class depends on $N \mod 8$ (Altland-Zirnbauer classification).

In [ ]:
# Level spacing ratio computation
N_values = [8, 10, 12, 14]
n_realizations = 50

cache_path = '../data/level_spacing_ratios.npz'
if check_cache(cache_path):
    data = load_results(cache_path)
    r_means = data['r_means']
    r_stds = data['r_stds']
    r_all_by_N = {N: data[f'r_all_N{N}'] for N in N_values}
    print('Loaded from cache.')
else:
    r_means = []
    r_stds = []
    r_all_by_N = {}
    
    for N in N_values:
        r_all = []
        for seed in range(n_realizations):
            syk = SYKHamiltonian(N, seed=seed, use_sparse=False)
            evals, _ = syk.diagonalize()
            r_vals, _ = level_spacing_ratio(evals)
            r_all.extend(r_vals)
        r_all = np.array(r_all)
        r_all_by_N[N] = r_all
        r_means.append(np.mean(r_all))
        r_stds.append(np.std(r_all) / np.sqrt(len(r_all)))
        print(f'N={N}: <r> = {r_means[-1]:.4f} +/- {r_stds[-1]:.4f} ({len(r_all)} ratios)')
    
    r_means = np.array(r_means)
    r_stds = np.array(r_stds)
    save_data = {'r_means': r_means, 'r_stds': r_stds, 'N_values': N_values}
    for N in N_values:
        save_data[f'r_all_N{N}'] = r_all_by_N[N]
    save_results(cache_path, **save_data)
    print('Saved to cache.')

In [ ]:
# Plot: <r> vs N
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: <r> vs N
ax = axes[0]
ax.errorbar(N_values, r_means, yerr=r_stds, fmt='o-', capsize=5, markersize=8, label='SYK q=4')
ax.axhline(y=0.5307, color='r', linestyle='--', label='GOE (0.5307)')
ax.axhline(y=0.6027, color='b', linestyle='--', label='GUE (0.6027)')
ax.axhline(y=0.3863, color='gray', linestyle=':', label='Poisson (0.3863)')
ax.set_xlabel('N (Majorana fermions)')
ax.set_ylabel(r'$\langle r \rangle$')
ax.set_title('Level Spacing Ratio vs System Size')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: histogram for N=12
ax = axes[1]
N_hist = 12
r_data = r_all_by_N[N_hist]
r_bins = np.linspace(0, 1, 50)
ax.hist(r_data, bins=r_bins, density=True, alpha=0.7, label=f'SYK N={N_hist}')

# GOE prediction: P(r) = 27(r+r^2)/(4(1+r+r^2)^{5/2})
r_th = np.linspace(0.01, 0.99, 200)
P_goe = 27 * (r_th + r_th**2) / (4 * (1 + r_th + r_th**2)**2.5)
P_gue = (81*np.sqrt(3))/(4*np.pi) * (r_th + r_th**2)**2 / (1 + r_th + r_th**2)**4
P_poisson = 2 / (1 + r_th)**2

ax.plot(r_th, P_goe, 'r-', linewidth=2, label='GOE')
ax.plot(r_th, P_gue, 'b-', linewidth=2, label='GUE')
ax.plot(r_th, P_poisson, 'gray', linestyle=':', linewidth=2, label='Poisson')
ax.set_xlabel('r')
ax.set_ylabel('P(r)')
ax.set_title(f'Level Spacing Ratio Distribution (N={N_hist})')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/01_level_spacing.png', dpi=150, bbox_inches='tight')
plt.show()
print('Level spacing analysis complete.')

## 2. Thermal Two-Point Function

The thermal two-point function $G(t) = \langle \psi_1(t) \psi_1(0) \rangle_\beta$ should exhibit conformal behavior $G(t) \sim 1/\sqrt{t}$ at large $\beta$ and intermediate times (the conformal regime).

Ref: Maldacena & Stanford, arXiv:1604.07818.

In [ ]:
# Two-point function computation
N_values_2pt = [10, 12, 14]
beta_values = [2, 4, 8, 16]
t_array = np.linspace(0.1, 30, 200)
n_real_2pt = 30

cache_path_2pt = '../data/two_point_functions.npz'
if check_cache(cache_path_2pt):
    data = load_results(cache_path_2pt)
    G_avg = {}
    for N in N_values_2pt:
        for beta in beta_values:
            key = f'G_N{N}_beta{beta}'
            if key in data:
                G_avg[(N, beta)] = data[key]
    print('Loaded two-point function data from cache.')
else:
    G_avg = {}
    for N in N_values_2pt:
        for beta in beta_values:
            G_all = []
            for seed in range(n_real_2pt):
                syk = SYKHamiltonian(N, seed=seed, use_sparse=False)
                evals, evecs = syk.diagonalize()
                psi_op = syk.majoranas[0]
                G = thermal_two_point_vectorized(evals, evecs, psi_op, beta, t_array)
                G_all.append(np.abs(G))
            G_avg[(N, beta)] = np.mean(G_all, axis=0)
            print(f'N={N}, beta={beta}: done')
    
    save_data = {'t_array': t_array}
    for (N, beta), G in G_avg.items():
        save_data[f'G_N{N}_beta{beta}'] = G
    save_results(cache_path_2pt, **save_data)
    print('Saved to cache.')

In [ ]:
# Plot two-point functions
fig, axes = plt.subplots(1, len(N_values_2pt), figsize=(5*len(N_values_2pt), 5))
if len(N_values_2pt) == 1:
    axes = [axes]

colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(beta_values)))

for ax, N in zip(axes, N_values_2pt):
    for beta, c in zip(beta_values, colors):
        key = (N, beta)
        if key in G_avg:
            ax.loglog(t_array, G_avg[key], color=c, label=f'$\\beta={beta}$')
    
    # Reference: 1/sqrt(t) line
    t_ref = t_array[t_array > 1]
    scale = G_avg.get((N, 16), G_avg.get((N, 8), np.ones_like(t_array)))
    ref_scale = scale[np.argmin(np.abs(t_array - 3))] * np.sqrt(3)
    ax.loglog(t_ref, ref_scale / np.sqrt(t_ref), 'k--', alpha=0.5, label=r'$\sim 1/\sqrt{t}$')
    
    ax.set_xlabel('t [1/J]')
    ax.set_ylabel('|G(t)|')
    ax.set_title(f'N={N}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Thermal Two-Point Function |G(t)| (disorder averaged)', y=1.02)
plt.tight_layout()
plt.savefig('../results/01_two_point.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. OTOC and Lyapunov Exponent

The out-of-time-order correlator (OTOC) $F(t) = \langle W(t) V W(t) V \rangle_\beta$ exhibits exponential growth at early times with Lyapunov exponent $\lambda_L$.

The chaos bound (MSS): $\lambda_L \leq 2\pi/\beta$.

SYK at large N saturates this bound: $\lambda_L = 2\pi/\beta$.
At finite N, we expect $\lambda_L < 2\pi/\beta$ with finite-size corrections.

Ref: Maldacena, Shenker & Stanford, arXiv:1503.01409.

In [ ]:
# OTOC computation
N_values_otoc = [10, 12, 14]
beta_values_otoc = [4, 8, 16]
t_otoc = np.linspace(0, 15, 150)
n_real_otoc = 30

cache_path_otoc = '../data/otoc.npz'
if check_cache(cache_path_otoc):
    data = load_results(cache_path_otoc)
    F_avg = {}
    lyap = {}
    for N in N_values_otoc:
        for beta in beta_values_otoc:
            key = f'F_N{N}_beta{beta}'
            if key in data:
                F_avg[(N, beta)] = data[key]
            lkey = f'lyap_N{N}_beta{beta}'
            if lkey in data:
                lyap[(N, beta)] = float(data[lkey])
    print('Loaded OTOC data from cache.')
else:
    F_avg = {}
    lyap = {}
    for N in N_values_otoc:
        for beta in beta_values_otoc:
            F_all = []
            for seed in range(n_real_otoc):
                syk = SYKHamiltonian(N, seed=seed, use_sparse=False)
                F = syk.otoc(beta, t_otoc, site_i=0, site_j=1)
                F_all.append(np.real(F))
            F_mean = np.mean(F_all, axis=0)
            F_avg[(N, beta)] = F_mean
            
            # Extract Lyapunov exponent
            lam, r2, fit_range = extract_lyapunov(F_mean, t_otoc)
            lyap[(N, beta)] = lam
            chaos_bound = 2 * np.pi / beta
            print(f'N={N}, beta={beta}: lambda_L = {lam:.4f}, bound = {chaos_bound:.4f}, ratio = {lam/chaos_bound:.3f}, R^2 = {r2:.3f}')
    
    save_data = {'t_otoc': t_otoc}
    for (N, beta), F in F_avg.items():
        save_data[f'F_N{N}_beta{beta}'] = F
    for (N, beta), l in lyap.items():
        save_data[f'lyap_N{N}_beta{beta}'] = l
    save_results(cache_path_otoc, **save_data)
    print('Saved to cache.')

In [ ]:
# Plot OTOC
fig, axes = plt.subplots(1, len(beta_values_otoc), figsize=(5*len(beta_values_otoc), 5))
colors_N = plt.cm.Set1(np.linspace(0, 0.5, len(N_values_otoc)))

for ax, beta in zip(axes, beta_values_otoc):
    for N, c in zip(N_values_otoc, colors_N):
        key = (N, beta)
        if key in F_avg:
            F0 = F_avg[key][0]
            decay = 1 - F_avg[key] / F0
            positive = decay > 1e-10
            ax.semilogy(t_otoc[positive], decay[positive], color=c, label=f'N={N}')
    
    # Chaos bound reference
    chaos_bound = 2 * np.pi / beta
    t_ref = np.linspace(0.5, 8, 100)
    ax.semilogy(t_ref, 0.01 * np.exp(chaos_bound * t_ref), 'k--', alpha=0.5,
                label=f'$e^{{2\\pi t/\\beta}}$')
    
    ax.set_xlabel('t [1/J]')
    ax.set_ylabel('1 - F(t)/F(0)')
    ax.set_title(f'$\\beta = {beta}$')
    ax.legend(fontsize=8)
    ax.set_ylim(1e-4, 10)
    ax.grid(True, alpha=0.3)

plt.suptitle('OTOC Decay (disorder averaged)', y=1.02)
plt.tight_layout()
plt.savefig('../results/01_otoc.png', dpi=150, bbox_inches='tight')
plt.show()

# Lyapunov summary
print('\nLyapunov exponent summary:')
print(f'{"N":>4} {"beta":>6} {"lambda_L":>10} {"2pi/beta":>10} {"ratio":>8}')
for N in N_values_otoc:
    for beta in beta_values_otoc:
        key = (N, beta)
        if key in lyap:
            bound = 2*np.pi/beta
            print(f'{N:>4} {beta:>6} {lyap[key]:>10.4f} {bound:>10.4f} {lyap[key]/bound:>8.3f}')

## 4. TFD Entanglement Entropy

The entanglement entropy $S(\beta) = -\text{Tr}(\rho_L \log \rho_L)$ of the TFD state should:
- Equal $\log(d)$ at $\beta = 0$ (maximally entangled)
- Decrease monotonically with $\beta$
- Approach 0 as $\beta \to \infty$ (ground state is product state for generic disorder)

In [ ]:
# TFD entanglement entropy
N_values_ent = [8, 10, 12]
beta_ent = np.array([0, 0.5, 1, 2, 4, 8, 16, 32])
n_real_ent = 30

cache_path_ent = '../data/tfd_entropy.npz'
if check_cache(cache_path_ent):
    data = load_results(cache_path_ent)
    S_avg = {N: data[f'S_N{N}'] for N in N_values_ent}
    S_err = {N: data[f'S_err_N{N}'] for N in N_values_ent}
    print('Loaded entropy data from cache.')
else:
    S_avg = {}
    S_err = {}
    for N in N_values_ent:
        dim_single = 2 ** (N // 2)
        S_all = []
        for seed in range(n_real_ent):
            ds = DoubledSYK(N, seed=seed, use_sparse=False)
            S_beta = []
            for beta in beta_ent:
                tfd, Z = build_tfd(ds, beta)
                S = entanglement_entropy(tfd, dim_single, dim_single)
                S_beta.append(S)
            S_all.append(S_beta)
        S_all = np.array(S_all)
        S_avg[N] = np.mean(S_all, axis=0)
        S_err[N] = np.std(S_all, axis=0) / np.sqrt(n_real_ent)
        print(f'N={N}: done')
    
    save_data = {'beta_ent': beta_ent}
    for N in N_values_ent:
        save_data[f'S_N{N}'] = S_avg[N]
        save_data[f'S_err_N{N}'] = S_err[N]
    save_results(cache_path_ent, **save_data)
    print('Saved to cache.')

In [ ]:
# Plot entanglement entropy
fig, ax = plt.subplots(figsize=(8, 6))
colors_N = plt.cm.Set1(np.linspace(0, 0.5, len(N_values_ent)))

for N, c in zip(N_values_ent, colors_N):
    dim_single = 2 ** (N // 2)
    S_max = np.log(dim_single)
    ax.errorbar(beta_ent, S_avg[N] / S_max, yerr=S_err[N] / S_max,
                fmt='o-', capsize=4, color=c, label=f'N={N} (S_max={S_max:.2f})')

ax.set_xlabel(r'$\beta$ [1/J]')
ax.set_ylabel(r'$S(\beta) / S_{\max}$')
ax.set_title('TFD Entanglement Entropy (disorder averaged)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.1)

plt.tight_layout()
plt.savefig('../results/01_tfd_entropy.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Spectral Form Factor

The spectral form factor $K(t) = |\text{Tr}(e^{-iHt})|^2 / d^2$ for chaotic systems shows the characteristic **dip-ramp-plateau** structure:
- **Dip** at early times (decay from K(0)=1)
- **Ramp** at intermediate times (linear growth, signature of RMT correlations)
- **Plateau** at late times (at $1/d$)

In [ ]:
# Spectral form factor
N_values_sff = [8, 10, 12]
n_real_sff = 50

cache_path_sff = '../data/spectral_form_factor.npz'
if check_cache(cache_path_sff):
    data = load_results(cache_path_sff)
    K_avg = {N: data[f'K_N{N}'] for N in N_values_sff}
    t_sff = data['t_sff']
    print('Loaded SFF data from cache.')
else:
    # Use logarithmic time spacing to capture the dip-ramp-plateau structure
    t_sff = np.logspace(-1, 4, 500)
    K_avg = {}
    
    for N in N_values_sff:
        K_all = []
        for seed in range(n_real_sff):
            syk = SYKHamiltonian(N, seed=seed, use_sparse=False)
            evals, _ = syk.diagonalize()
            K = spectral_form_factor(evals, t_sff)
            K_all.append(K)
        K_avg[N] = np.mean(K_all, axis=0)
        print(f'N={N}: done')
    
    save_data = {'t_sff': t_sff}
    for N in N_values_sff:
        save_data[f'K_N{N}'] = K_avg[N]
    save_results(cache_path_sff, **save_data)
    print('Saved to cache.')

In [ ]:
# Plot spectral form factor
fig, ax = plt.subplots(figsize=(10, 6))
colors_N = plt.cm.Set1(np.linspace(0, 0.5, len(N_values_sff)))

for N, c in zip(N_values_sff, colors_N):
    dim = 2 ** (N // 2)
    ax.loglog(t_sff, K_avg[N], color=c, label=f'N={N} (d={dim})', alpha=0.8)
    ax.axhline(y=1.0/dim, color=c, linestyle=':', alpha=0.5)

ax.set_xlabel('t [1/J]')
ax.set_ylabel('K(t) / d$^2$')
ax.set_title('Spectral Form Factor (disorder averaged, 50 realizations)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(1e-4, 2)

plt.tight_layout()
plt.savefig('../results/01_sff.png', dpi=150, bbox_inches='tight')
plt.show()
print('Spectral form factor analysis complete.')

## Foundation Gate Summary

All foundation benchmarks should satisfy:
1. Level spacing ratio in RMT range (GOE/GUE)
2. Two-point function shows conformal regime at large beta
3. Lyapunov exponent approaches but does not exceed chaos bound
4. TFD entropy = log(d) at beta=0, decreasing monotonically
5. Spectral form factor shows dip-ramp-plateau structure

In [ ]:
# Foundation gate check
print('=== Foundation Gate Check ===')
print()

# 1. Level spacing
for i, N in enumerate(N_values):
    status = 'PASS' if 0.45 < r_means[i] < 0.65 else 'FAIL'
    print(f'Level spacing N={N}: <r>={r_means[i]:.4f} [{status}]')

# 2. Two-point function (visual check - just report completion)
print(f'Two-point function: computed for N={N_values_2pt}, beta={beta_values}')

# 3. Lyapunov
for N in N_values_otoc:
    for beta in beta_values_otoc:
        key = (N, beta)
        if key in lyap:
            bound = 2*np.pi/beta
            status = 'PASS' if lyap[key] <= bound * 1.1 else 'WARN'  # allow 10% for finite-size
            print(f'Lyapunov N={N}, beta={beta}: lambda={lyap[key]:.4f}, bound={bound:.4f} [{status}]')

# 4. Entropy
for N in N_values_ent:
    S_max = np.log(2 ** (N // 2))
    s0 = S_avg[N][0]
    status = 'PASS' if abs(s0 - S_max) < 0.01 else 'FAIL'
    print(f'TFD entropy N={N}: S(beta=0)={s0:.4f}, S_max={S_max:.4f} [{status}]')

print()
print('Foundation verification complete.')